# FireSight-IR | Module 1a - VIIRS & FIRMS Cloud Ingestion (v2)

**Project:** FireSight-IR - Physics-constrained wildfire intelligence at constellation scale  
**Platform:** Google Colab (cloud-native, no local downloads)  
**Version:** v2.0 - incorporates post-review corrections  

---

## Corrections from v1 review

| # | Issue | Fix applied |
|---|---|---|
| 1 | 🔴 Deduplication dropped valid multi-overpass fire pixels | Changed key to `['latitude', 'longitude', 'granule_id']` |
| 2 | 🔴 Background variables not inspected for fill values | Added empirical fill-value audit before extraction; conditional masking |
| 3 | 🟡 BTD verification threshold too loose (`>0`) | Updated to `>15 K` - the documented nominal-confidence boundary |
| 4 | 🟡 Silent `date='unknown'` fallback with no warning | Now warns explicitly and flags affected pixels post-extraction |
| 5 | 🟢 Missing UTC timestamp - needed for ERA5 co-location in Module 1b | `datetime_utc` column now extracted from granule metadata |
| 6 | 🟢 `confidence` encoding unverified (fire mask class vs FP_confidence) | Structure audit now explicitly checks encoding of `FP_confidence` |

---

## What this notebook does

1. **Mounts Google Drive** for persistent output storage across Colab sessions
2. **Downloads FIRMS** VIIRS S-NPP Standard Processing fire detection archives (2018–2023) via the FIRMS area API
3. **Runs a structure + fill-value audit** on one VNP14IMG v002 granule before batch extraction
4. **Streams VNP14IMG** Collection 2 granules from NASA Earthdata S3 - guided by FIRMS active-fire days
5. **Extracts fire pixel records** with correct deduplication, fill masking, UTC timestamps, and derived features
6. **Verifies output quality** against expected statistics

> **Note:** 32×32 spatial patch extraction happens in Module 1c. This notebook produces the per-pixel tabular dataset that feeds Modules 1b (ERA5 co-location) and 2 (feature engineering).

---

## Before running

You need two free accounts:
- **NASA Earthdata:** https://urs.earthdata.nasa.gov/users/new
- **FIRMS map key:** https://firms.modaps.eosdis.nasa.gov/api/map_key/

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


---
## Section 0 - Install packages and mount Drive

**Run this section at the start of every Colab session.** Packages are wiped on disconnect.

In [13]:
# ── Install packages (run every session) ─────────────────────────────────────
!pip install earthaccess requests pandas numpy xarray h5py h5netcdf pyarrow tqdm -q
print("✓ Packages installed")

✓ Packages installed


In [14]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted at /content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive mounted at /content/drive


In [15]:
# ── Core imports ──────────────────────────────────────────────────────────────
import os
import time
import warnings
import requests
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import xarray as xr
import h5py
import earthaccess
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')  # suppress xarray phony_dims warnings
print("✓ Imports complete")

✓ Imports complete


---
## Section 1 - Project configuration

All parameters live here. **Do not hardcode values anywhere else in the notebook.**

In [16]:
# ── Spatial domain - Western US (CA + NV + OR + parts of WA, ID, AZ, UT) ─────
# Three formats required because different libraries use different conventions

BBOX_TUPLE = (-125, 32, -109, 49)               # (west, south, east, north) - earthaccess
BBOX_STR   = '-125,32,-109,49'                  # 'west,south,east,north'    - FIRMS API URL
BBOX_DICT  = {                                   # dict of floats             - DataFrame filtering
    'lon_min': -125, 'lon_max': -109,
    'lat_min':   32, 'lat_max':   49,
}
# CRITICAL: If you change one format, change all three.

# ── Temporal domain ───────────────────────────────────────────────────────────
TRAIN_YEARS        = [2018, 2019, 2020, 2021, 2022]  # training data
VAL_YEAR           = 2023                             # held-out validation - never touch during training
ALL_YEARS          = TRAIN_YEARS + [VAL_YEAR]
FIRE_SEASON_MONTHS = [5, 6, 7, 8, 9, 10]             # May–Oct: >95% of western US fire activity

# ── Data sources ──────────────────────────────────────────────────────────────
FIRMS_SOURCE       = 'VIIRS_SNPP_SP'   # Standard Processing - quality-controlled archive
                                        # Do NOT use 'VIIRS_SNPP_NRT' (preliminary product)
VIIRS_SHORT_NAME   = 'VNP14IMG'        # VIIRS/NPP I-Band 375m Active Fire, Collection 2
VIIRS_VERSION      = '002'             # Collection 2 - always use this, not 001

# ── FIRMS-guided VIIRS download filter ───────────────────────────────────────
# Only process VIIRS granules on days with at least this many FIRMS detections.
# Eliminates ~85% of granules with no significant fire activity.
MIN_FIRMS_COUNT = 10

# ── Your FIRMS map key ────────────────────────────────────────────────────────
# Get from: https://firms.modaps.eosdis.nasa.gov/api/map_key/
FIRMS_MAP_KEY = 'YOUR_FIRMS_MAP_KEY_HERE'  # <-- replace this

# ── Output paths on Google Drive ─────────────────────────────────────────────
DRIVE_ROOT  = Path('/content/drive/MyDrive/firesight-ir')
FIRMS_DIR   = DRIVE_ROOT / 'data/raw/firms'
VIIRS_DIR   = DRIVE_ROOT / 'data/processed/viirs_fp'
AUDIT_DIR   = DRIVE_ROOT / 'data/audit'
FIGURES_DIR = DRIVE_ROOT / 'figures'

for d in [FIRMS_DIR, VIIRS_DIR, AUDIT_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("✓ Configuration set")
print(f"  Training years : {TRAIN_YEARS}")
print(f"  Validation year: {VAL_YEAR}  (held out - do not use for training)")
print(f"  Bounding box   : {BBOX_STR}")
print(f"  FIRMS source   : {FIRMS_SOURCE}")
print(f"  VIIRS product  : {VIIRS_SHORT_NAME} v{VIIRS_VERSION}")
print(f"  Drive root     : {DRIVE_ROOT}")

✓ Configuration set
  Training years : [2018, 2019, 2020, 2021, 2022]
  Validation year: 2023  (held out - do not use for training)
  Bounding box   : -125,32,-109,49
  FIRMS source   : VIIRS_SNPP_SP
  VIIRS product  : VNP14IMG v002
  Drive root     : /content/drive/MyDrive/firesight-ir


---
## Section 2 - NASA Earthdata authentication

**Run every session.** Credentials are not persisted in Colab.

In [17]:
# ── Authenticate to NASA Earthdata ────────────────────────────────────────────
auth = earthaccess.login(strategy='interactive')

assert auth.authenticated, (
    "Authentication failed. "
    "Verify credentials at https://urs.earthdata.nasa.gov"
)
print(f"✓ Authenticated as: {auth.username}")
print("✓ S3 direct access enabled - no files will be written to Colab disk")

✓ Authenticated as: Msledge7
✓ S3 direct access enabled - no files will be written to Colab disk


---
## Section 3 - FIRMS download

### How the FIRMS API works

The FIRMS area API returns fire detections as CSV for a bounding box, time period, and source.
The URL structure is:
```
https://firms.modaps.eosdis.nasa.gov/api/area/csv/{MAP_KEY}/{SOURCE}/{BBOX}/{DAY_RANGE}/{DATE}
```
**`DAY_RANGE` is capped at 5.** The `fetch_firms_year()` function loops in 5-day increments automatically.

In [18]:
# ── FIRMS fetch function ───────────────────────────────────────────────────────
FIRMS_BASE = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv'

def fetch_firms_year(
    year: int,
    map_key: str,
    source: str,
    bbox_str: str,
    bbox_dict: dict,
    fire_season_months: list,
    day_chunk: int = 5,
    retry_delay: float = 2.0,
    max_retries: int = 3,
) -> pd.DataFrame:
    """
    Download a full year of FIRMS fire detections for a bounding box.

    Loops through the year in `day_chunk`-day windows (max 5, API limit).
    Filters to fire_season_months after concatenation.
    Returns a single DataFrame with all detections.

    Parameters
    ----------
    year               : calendar year (int)
    map_key            : FIRMS API map key (str)
    source             : FIRMS source string, e.g. 'VIIRS_SNPP_SP'
    bbox_str           : 'west,south,east,north' string
    bbox_dict          : dict with lon_min, lon_max, lat_min, lat_max
    fire_season_months : list of month ints to retain (e.g. [5,6,7,8,9,10])
    day_chunk          : days per API call (max 5)
    retry_delay        : seconds to wait before retrying on error
    max_retries        : maximum retries per chunk

    Returns
    -------
    pd.DataFrame with FIRMS columns plus 'month' column
    """
    assert day_chunk <= 5, "FIRMS API day_range cap is 5 - do not exceed"

    start_date = pd.Timestamp(f'{year}-01-01')
    end_date   = pd.Timestamp(f'{year}-12-31')
    chunks = pd.date_range(start=start_date, end=end_date, freq=f'{day_chunk}D')

    frames = []
    failed_chunks = []

    for chunk_start in tqdm(chunks, desc=f'FIRMS {year}', unit='chunk', leave=False):
        date_str = chunk_start.strftime('%Y-%m-%d')
        url = f"{FIRMS_BASE}/{map_key}/{source}/{bbox_str}/{day_chunk}/{date_str}"

        for attempt in range(max_retries):
            try:
                resp = requests.get(url, timeout=30)
                if resp.status_code == 200 and len(resp.text.strip()) > 0:
                    df = pd.read_csv(StringIO(resp.text))
                    if len(df) > 0 and 'latitude' in df.columns:
                        frames.append(df)
                    break
                elif resp.status_code == 400:
                    # 400 often means empty result for this date range - not an error
                    break
                else:
                    if attempt < max_retries - 1:
                        time.sleep(retry_delay)
            except requests.RequestException:
                if attempt < max_retries - 1:
                    time.sleep(retry_delay)
                else:
                    failed_chunks.append(date_str)

    if not frames:
        print(f"  [WARN] {year}: no data returned - check map key and source")
        return pd.DataFrame()

    result = pd.concat(frames, ignore_index=True)

    # Parse date and filter to fire season
    result['acq_date'] = pd.to_datetime(result['acq_date'])
    result['month']    = result['acq_date'].dt.month
    result = result[result['month'].isin(fire_season_months)].copy()

    # Bounding box filter (FIRMS returns slight overestimates at bbox edges)
    result = result[
        (result['longitude'] >= bbox_dict['lon_min']) &
        (result['longitude'] <= bbox_dict['lon_max']) &
        (result['latitude']  >= bbox_dict['lat_min']) &
        (result['latitude']  <= bbox_dict['lat_max'])
    ].copy()

    if failed_chunks:
        print(f"  [WARN] {year}: {len(failed_chunks)} chunks failed: {failed_chunks[:5]}")

    return result


print("✓ fetch_firms_year() defined")

✓ fetch_firms_year() defined


In [19]:
# ── Download FIRMS for all years ──────────────────────────────────────────────
# Skips years already saved to Drive - safe to rerun after Colab disconnect.

assert FIRMS_MAP_KEY != 'f97281ae8847f65c02ab6b2c9ac60c13', (
    "Set your FIRMS map key in the config cell (Section 1)"
)

firms_summary = {}

for year in ALL_YEARS:
    out_path = FIRMS_DIR / f'firms_viirs_snpp_{year}.parquet'

    if out_path.exists():
        df = pd.read_parquet(out_path)
        print(f"  {year}: {len(df):>9,} detections  [already on Drive - skipped]")
        firms_summary[year] = len(df)
        continue

    print(f"── {year} ──")
    df = fetch_firms_year(
        year               = year,
        map_key            = FIRMS_MAP_KEY,
        source             = FIRMS_SOURCE,
        bbox_str           = BBOX_STR,
        bbox_dict          = BBOX_DICT,
        fire_season_months = FIRE_SEASON_MONTHS,
    )

    if len(df) > 0:
        df.to_parquet(out_path, index=False)
        print(f"  {year}: {len(df):>9,} detections saved → Drive")
        firms_summary[year] = len(df)
    else:
        print(f"  {year}: no data - check map key")
        firms_summary[year] = 0

print("\n✓ FIRMS download complete")

  2018:   197,013 detections  [already on Drive - skipped]
  2019:    62,576 detections  [already on Drive - skipped]
  2020:   358,026 detections  [already on Drive - skipped]
  2021:   357,975 detections  [already on Drive - skipped]
  2022:    98,049 detections  [already on Drive - skipped]
  2023:    76,084 detections  [already on Drive - skipped]

✓ FIRMS download complete


In [20]:
# ── Inspect FIRMS output ──────────────────────────────────────────────────────
# Load 2020 (peak fire year) and inspect columns, value ranges, type distribution

firms_2020_path = FIRMS_DIR / 'firms_viirs_snpp_2020.parquet'
if firms_2020_path.exists():
    df20 = pd.read_parquet(firms_2020_path)

    print(f"2020 FIRMS: {len(df20):,} rows × {len(df20.columns)} columns")
    print(f"\nColumns: {list(df20.columns)}")
    print(f"\nDate range: {df20['acq_date'].min()} → {df20['acq_date'].max()}")

    # Type distribution - critical for label construction
    print("\ntype column distribution:")
    print(df20['type'].value_counts().to_string())
    print("  → type=0: vegetation fire (positive wildfire label)")
    print("  → type=2: static land source - gas flares, industrial (false-alarm training)")
    print("  → type=3: offshore (exclude from training)")

    print(f"\nBT_I4 (bright_ti4) range: {df20['bright_ti4'].min():.1f} – {df20['bright_ti4'].max():.1f} K")
    print(f"FRP range: {df20['frp'].min():.2f} – {df20['frp'].max():.1f} MW")
    print(f"\nconfidence distribution:")
    print(df20['confidence'].value_counts().to_string())
else:
    print("2020 FIRMS file not found - check download completed above")

2020 FIRMS: 358,026 rows × 16 columns

Columns: ['latitude', 'longitude', 'bright_ti4', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_ti5', 'frp', 'daynight', 'type', 'month']

Date range: 2020-05-01 00:00:00 → 2020-10-31 00:00:00

type column distribution:
type
0    350071
2      7555
3       400
  → type=0: vegetation fire (positive wildfire label)
  → type=2: static land source - gas flares, industrial (false-alarm training)
  → type=3: offshore (exclude from training)

BT_I4 (bright_ti4) range: 208.0 – 367.0 K
FRP range: 0.00 – 2110.7 MW

confidence distribution:
confidence
n    309057
h     30812
l     18157


---
## Section 4 - FIRMS-guided active fire day selection

In [21]:
def get_active_fire_days(year: int, firms_dir: Path, min_count: int = 10) -> list:
    """
    Return a sorted list of date strings ('YYYY-MM-DD') for days with
    at least `min_count` FIRMS fire detections in the study region.

    Parameters
    ----------
    year      : calendar year
    firms_dir : Path to directory containing firms_viirs_snpp_{year}.parquet
    min_count : minimum FIRMS detections to qualify as an active fire day

    Returns
    -------
    list of 'YYYY-MM-DD' strings, sorted ascending
    """
    path = firms_dir / f'firms_viirs_snpp_{year}.parquet'
    if not path.exists():
        print(f"  [WARN] FIRMS file missing for {year}: {path}")
        return []

    df = pd.read_parquet(path, columns=['acq_date', 'latitude'])
    df['acq_date'] = pd.to_datetime(df['acq_date']).dt.strftime('%Y-%m-%d')

    day_counts = df.groupby('acq_date').size()
    active_days = sorted(day_counts[day_counts >= min_count].index.tolist())

    return active_days


# Preview active fire day counts per year
print(f"Active fire day selection (MIN_FIRMS_COUNT = {MIN_FIRMS_COUNT})")
print(f"{'Year':<8} {'Active days':<14} {'Skipped days':<14}")
print('─' * 40)

for year in ALL_YEARS:
    active = get_active_fire_days(year, FIRMS_DIR, MIN_FIRMS_COUNT)
    season_days = len(FIRE_SEASON_MONTHS) * 30
    skipped = season_days - len(active)
    flag = '← validation (do not use for training)' if year == VAL_YEAR else ''
    print(f"{year:<8} {len(active):<14} {skipped:<14} {flag}")

print("✓ FIRMS-guided day selection defined")

Active fire day selection (MIN_FIRMS_COUNT = 10)
Year     Active days    Skipped days  
────────────────────────────────────────
2018     184            -4             
2019     184            -4             
2020     184            -4             
2021     184            -4             
2022     169            11             
2023     183            -3             ← validation (do not use for training)
✓ FIRMS-guided day selection defined


---
## Section 5 - VNP14IMG structure & fill-value audit

**Run this section before defining `FP_VARS` or `FP_FILL`.**  
VNP14IMG v002 variable paths and fill value behaviour must be verified empirically - 
they differ from v001 and from the MODIS heritage product. This audit drives the 
configuration in Section 6.

> **FIX #2 from review:** Background variables (`FP_MeanT4`, `FP_MeanT5`, `FP_MeanDT`, 
> `FP_MAD_T4`, `FP_MAD_T5`) may use native `NaN` rather than a `-999` sentinel. 
> The audit below checks the actual `_FillValue` attribute for every variable before 
> the extractor is configured.

In [22]:
# ── Open one granule and walk the HDF5 tree ───────────────────────────────────
# Pull from a confirmed high-fire day: 2020-09-05 (Creek Fire ignition)

AUDIT_DATE = '2020-09-05'

audit_granules = earthaccess.search_data(
    short_name   = VIIRS_SHORT_NAME,
    version      = VIIRS_VERSION,
    temporal     = (AUDIT_DATE, AUDIT_DATE),
    bounding_box = BBOX_TUPLE,
    count        = 2,
)

print(f"Found {len(audit_granules)} granules on {AUDIT_DATE}")

if audit_granules:
    print(f"Opening: {audit_granules[0]['meta']['native-id']}\n")

    def walk_h5(h5file, prefix='', depth=0, max_depth=4):
        if depth >= max_depth:
            return
        for key in h5file.keys():
            item = h5file[key]
            path = f"{prefix}/{key}"
            if isinstance(item, h5py.Dataset):
                print(f"  {'DATASET':<10} {path:<65} shape={item.shape}  dtype={item.dtype}")
            else:
                print(f"  {'GROUP':<10} {path}")
                walk_h5(item, path, depth + 1, max_depth)

    with earthaccess.open([audit_granules[0]]) as fhs:
        with h5py.File(fhs[0], 'r') as hf:
            print("═══ VNP14IMG v002 HDF5 structure ═══")
            walk_h5(hf)
            print("\n═══ Global attributes (first 12) ═══")
            for k in list(hf.attrs)[:12]:
                print(f"  {k}: {hf.attrs[k]}")

Found 2 granules on 2020-09-05
Opening: VNP14IMG.A2020249.0818.002.2024066054445



QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

TypeError: 'list' object does not support the context manager protocol

In [ ]:
# ── Fill-value audit for Fire Pixel variables ─────────────────────────────────
# For each variable we plan to read, check:
#   (a) the _FillValue attribute - tells us if a sentinel exists
#   (b) the actual min/max in this granule - sanity check
#
# FIX #2: Do NOT assume -999 for background variables. Read the attribute.
# If _FillValue is absent ('filling off'), the variable uses NaN natively -
# no sentinel masking needed, but NaN propagation must still be handled.

CANDIDATE_PATHS = {
    'latitude':   'Fire Pixels/FP_latitude',
    'longitude':  'Fire Pixels/FP_longitude',
    'BT_I4':      'Fire Pixels/FP_T4',
    'BT_I5':      'Fire Pixels/FP_T5',
    'BT_I4_bg':   'Fire Pixels/FP_MeanT4',
    'BT_I5_bg':   'Fire Pixels/FP_MeanT5',
    'MAD_I4':     'Fire Pixels/FP_MAD_T4',
    'MAD_I5':     'Fire Pixels/FP_MAD_T5',
    'frp_mw':     'Fire Pixels/FP_power',
    'confidence': 'Fire Pixels/FP_confidence',
    'sol_zen':    'Fire Pixels/FP_SolZenAng',
    'view_zen':   'Fire Pixels/FP_ViewZenAng',
    'BT_diff_bg': 'Fire Pixels/FP_MeanDT',
    'is_day':     'Fire Pixels/FP_day',
}

print("═══ Fill-value audit for Fire Pixel variables ═══\n")
print(f"{'Column':<14} {'HDF5 path':<35} {'_FillValue':<18} {'dtype':<10} {'min':>10} {'max':>10}")
print('─' * 105)

DISCOVERED_FILL = {}  # will be populated and used in Section 6

if audit_granules:
    with earthaccess.open([audit_granules[0]]) as fhs:
        with h5py.File(fhs[0], 'r') as hf:
            for col, path in CANDIDATE_PATHS.items():
                try:
                    ds = hf[path]
                    arr = ds[:].astype(np.float32)

                    # Check for _FillValue attribute
                    fill_attr = ds.attrs.get('_FillValue', None)
                    if fill_attr is not None:
                        fill_val = float(fill_attr)
                        DISCOVERED_FILL[col] = fill_val
                        fill_str = str(fill_val)
                    else:
                        fill_str = 'NaN native'
                        # No sentinel - NaN is native; no masking needed

                    valid = arr[~np.isnan(arr)]
                    if fill_attr is not None:
                        valid = valid[valid != fill_val]
                    min_v = f"{valid.min():.2f}" if len(valid) > 0 else 'empty'
                    max_v = f"{valid.max():.2f}" if len(valid) > 0 else 'empty'

                    print(f"{col:<14} {path:<35} {fill_str:<18} {str(ds.dtype):<10} {min_v:>10} {max_v:>10}")

                except KeyError:
                    print(f"{col:<14} {path:<35} {'PATH NOT FOUND':^40}  ← UPDATE CANDIDATE_PATHS")

print(f"\n✓ Fill-value audit complete")
print(f"  Variables with sentinel fill: {list(DISCOVERED_FILL.keys())}")
print(f"  Variables using native NaN:   {[k for k in CANDIDATE_PATHS if k not in DISCOVERED_FILL]}")

In [ ]:
# ── FP_confidence encoding audit ──────────────────────────────────────────────
# FIX #6: The notebook previously assumed FP_confidence encodes as 8=nominal,
# 9=high (fire mask SDS class values). The fire mask classes 7/8/9 and the
# FP_confidence sparse array may use different encodings.
#
# Documented nominal-confidence definition (NASA/FIRMS):
#   Nominal = free of sun glint + BT anomaly > 15 K in I4
#   High     = day or nighttime I4 saturated pixels
#
# This cell prints the unique values and distribution of FP_confidence
# so you can confirm the encoding before using it as a filter.

print("═══ FP_confidence encoding audit ═══\n")

if audit_granules:
    with earthaccess.open([audit_granules[0]]) as fhs:
        with h5py.File(fhs[0], 'r') as hf:
            try:
                conf_path = CANDIDATE_PATHS['confidence']
                conf_arr = hf[conf_path][:]
                unique_vals, counts = np.unique(conf_arr, return_counts=True)

                print(f"FP_confidence dtype : {conf_arr.dtype}")
                print(f"Unique values       : {unique_vals.tolist()}")
                print()
                print(f"{'Value':<10} {'Count':<10} {'Interpretation (verify below)'}")
                print('─' * 55)
                for v, c in zip(unique_vals, counts):
                    # Fire mask classes: 7=low, 8=nominal, 9=high
                    # Some versions use 0/1/2 or 'l'/'n'/'h' - confirm here
                    if v == 7:
                        interp = 'low confidence (fire mask class)'
                    elif v == 8:
                        interp = 'nominal confidence (fire mask class)'
                    elif v == 9:
                        interp = 'high confidence (fire mask class)'
                    else:
                        interp = '← UNEXPECTED - check User Guide Table 3'
                    print(f"  {v:<8} {c:<10} {interp}")

                print()
                print("  Expected encoding for VNP14IMG v002: 7=low, 8=nominal, 9=high")
                print("  If values differ, update the CONFIDENCE_NOMINAL and CONFIDENCE_HIGH")
                print("  constants in Section 6 before running extraction.")

            except KeyError:
                print(f"  [ERROR] FP_confidence not found at {conf_path}")
                print("  Update CANDIDATE_PATHS['confidence'] to the correct path.")

---
## Section 6 - Variable paths and fill-value configuration

**Update the values below based on the audit output in Section 5.**  
`FP_VARS` paths must match exactly what the structure audit showed.  
`FP_FILL` is populated from `DISCOVERED_FILL` automatically - you only need to 
manually override if the audit shows a sentinel other than what was detected.

In [ ]:
# ── VNP14IMG v002 variable paths ──────────────────────────────────────────────
# Populated from CANDIDATE_PATHS - update if structure audit showed different paths.

FP_VARS = CANDIDATE_PATHS.copy()

# ── Fill values - driven by empirical audit, not assumptions ──────────────────
# DISCOVERED_FILL is built by Section 5. It only contains variables that
# actually have a _FillValue attribute. Variables using native NaN (filling off)
# are NOT included - they do not need sentinel masking.
#
# FIX #2: Background variables (BT_I4_bg, BT_I5_bg, MAD_I4, MAD_I5,
# BT_diff_bg) are only added to FP_FILL if the audit found a sentinel for them.
# Do NOT assume -999 - the v002 product may use NaN natively for these.

FP_FILL = DISCOVERED_FILL.copy()

# ── Confidence encoding constants ─────────────────────────────────────────────
# Confirm these match the FP_confidence audit output above.
# Default: 7=low, 8=nominal, 9=high (VNP14IMG fire mask class encoding)
CONFIDENCE_LOW     = 7
CONFIDENCE_NOMINAL = 8
CONFIDENCE_HIGH    = 9

print("✓ FP_VARS configured")
print(f"  {len(FP_VARS)} variables defined")
print()
print("✓ FP_FILL configured from audit")
print(f"  Variables with sentinel masking : {list(FP_FILL.keys())}")
print(f"  Variables using native NaN      : {[k for k in FP_VARS if k not in FP_FILL]}")
print()
print(f"  Confidence encoding: {CONFIDENCE_LOW}=low | {CONFIDENCE_NOMINAL}=nominal | {CONFIDENCE_HIGH}=high")
print("  → Update CONFIDENCE_* constants above if audit showed different values")

---
## Section 7 - Single-granule fire pixel extractor

Key corrections from v1:
- **FIX #4:** UTC datetime extracted from granule metadata; explicit warning if parsing fails
- **FIX #2:** Fill masking driven by empirical `FP_FILL` (audit-derived), not hardcoded assumptions
- **FIX #7:** `BTD_zscore` added as MAD-normalized spectral contrast feature

In [ ]:
def _parse_granule_datetime(granule) -> tuple[str, str]:
    """
    Extract UTC datetime and date string from granule UMM-G metadata.

    Returns (datetime_utc_str, date_str). Both are 'unknown' on failure,
    with an explicit warning printed.

    FIX #4: v1 silently returned 'unknown' with no warning, causing rows
    with date='unknown' to be silently excluded from active-fire-day grouping.
    """
    granule_id = str(granule.get('meta', {}).get('native-id', 'unknown'))
    try:
        time_str = (
            granule['umm']['TemporalExtent']['RangeDateTime']['BeginningDateTime']
        )
        datetime_utc = time_str[:19]  # 'YYYY-MM-DDTHH:MM:SS'
        date_str     = time_str[:10]  # 'YYYY-MM-DD'
        return datetime_utc, date_str

    except (KeyError, TypeError, IndexError):
        print(
            f"  [WARN] Could not parse datetime from granule metadata: {granule_id}\n"
            f"         Rows from this granule will have datetime_utc='unknown'. "
            f"         These rows will be excluded from ERA5 co-location in Module 1b."
        )
        return 'unknown', 'unknown'


def extract_granule_fp(
    file_handle,
    granule,
    bbox_dict: dict,
    fp_vars: dict,
    fp_fill: dict,
) -> pd.DataFrame:
    """
    Extract fire pixel records from a single VNP14IMG granule.

    Reads fire pixel arrays (phony_dim_0 dimension), applies spatial
    filtering, masks sentinel fill values per the audit-derived FP_FILL
    dict, and computes derived features.

    Parameters
    ----------
    file_handle : S3 streaming file object from earthaccess.open()
    granule     : earthaccess DataGranule object (for metadata)
    bbox_dict   : dict with lon_min, lon_max, lat_min, lat_max
    fp_vars     : dict mapping output column names to HDF5 paths
    fp_fill     : dict mapping column names to fill value sentinels.
                  Only variables present in this dict are sentinel-masked.
                  Variables absent from it are assumed to use native NaN.

    Returns
    -------
    pd.DataFrame with one row per fire pixel in bbox, or empty DataFrame.

    Columns
    -------
    All fp_vars keys + BTD, BT_I4_anom, BTD_anom, BTD_zscore,
    datetime_utc, date, granule_id.
    """
    records = {}

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        with h5py.File(file_handle, 'r') as hf:

            # ── Check fire pixel count ────────────────────────────────────────
            try:
                lat_arr = hf[fp_vars['latitude']][:].astype(np.float32)
            except KeyError:
                return pd.DataFrame()  # path mismatch - update FP_VARS

            if lat_arr.size == 0:
                return pd.DataFrame()

            lon_arr = hf[fp_vars['longitude']][:].astype(np.float32)

            # ── Spatial filter ────────────────────────────────────────────────
            in_bbox = (
                (lon_arr >= bbox_dict['lon_min']) & (lon_arr <= bbox_dict['lon_max']) &
                (lat_arr >= bbox_dict['lat_min']) & (lat_arr <= bbox_dict['lat_max'])
            )

            if not in_bbox.any():
                return pd.DataFrame()

            records['latitude']  = lat_arr[in_bbox]
            records['longitude'] = lon_arr[in_bbox]

            # ── Read remaining variables ──────────────────────────────────────
            for col, h5path in fp_vars.items():
                if col in ('latitude', 'longitude'):
                    continue
                try:
                    arr = hf[h5path][:].astype(np.float32)[in_bbox]

                    # FIX #2: Only mask sentinel if this variable has one.
                    # Background variables may use native NaN - no masking needed.
                    if col in fp_fill:
                        arr = np.where(arr == fp_fill[col], np.nan, arr)

                    records[col] = arr
                except KeyError:
                    records[col] = np.full(in_bbox.sum(), np.nan)

    # ── Build DataFrame ───────────────────────────────────────────────────────
    df = pd.DataFrame(records)

    # ── Derived features ──────────────────────────────────────────────────────
    # BTD: primary fire spectral signal (MWIR − LWIR). Fire >> 0.
    df['BTD'] = df['BT_I4'] - df['BT_I5']

    # BT_I4_anom: fire intensity above local ambient surface temperature.
    df['BT_I4_anom'] = df['BT_I4'] - df['BT_I4_bg']

    # BTD_anom: fire spectral contrast above background spectral contrast.
    # Suppresses persistent warm surfaces (deserts, UHI) that inflate BTD.
    df['BTD_anom'] = df['BTD'] - df['BT_diff_bg']

    # BTD_zscore: MAD-normalised spectral contrast.
    # More physically meaningful than raw BTD_anom for ML input because it
    # normalises for seasonal and geographic variation in background MAD.
    # 1e-6 floor prevents division by zero on uniform background patches.
    df['BTD_zscore'] = (df['BTD'] - df['BT_diff_bg']) / (df['MAD_I4'] + 1e-6)

    # ── UTC timestamp - FIX #5 ────────────────────────────────────────────────
    # Module 1b ERA5 co-location requires UTC hour resolution.
    # Extract full timestamp now rather than retrofitting in Module 1b.
    datetime_utc, date_str = _parse_granule_datetime(granule)
    df['datetime_utc'] = datetime_utc  # 'YYYY-MM-DDTHH:MM:SS' or 'unknown'
    df['date']         = date_str      # 'YYYY-MM-DD' or 'unknown'

    # ── Granule ID for deduplication ─────────────────────────────────────────
    df['granule_id'] = str(granule.get('meta', {}).get('native-id', 'unknown'))

    return df


print("✓ extract_granule_fp() defined")
print("  Derived columns: BTD, BT_I4_anom, BTD_anom, BTD_zscore")
print("  Metadata columns: datetime_utc, date, granule_id")

---
## Section 8 - Quick single-day test (2020-09-05)

**Always run this before the full batch.** Verify fire pixel counts and value ranges 
look reasonable before committing to a 90–120 minute extraction run.

**FIX #3:** BTD verification threshold updated from `>0` to `>15 K`, which is the 
documented boundary for nominal-confidence pixels per the NASA/FIRMS algorithm guide.

In [ ]:
# ── Test extraction on Creek Fire ignition day ─────────────────────────────
TEST_DATE = '2020-09-05'

test_granules = earthaccess.search_data(
    short_name   = VIIRS_SHORT_NAME,
    version      = VIIRS_VERSION,
    temporal     = (TEST_DATE, TEST_DATE),
    bounding_box = BBOX_TUPLE,
    count        = 10,
)

print(f"Found {len(test_granules)} granules on {TEST_DATE}")

test_frames = []
with earthaccess.open(test_granules[:5]) as fhs:
    for granule, fh in zip(test_granules[:5], fhs):
        df = extract_granule_fp(fh, granule, BBOX_DICT, FP_VARS, FP_FILL)
        if len(df) > 0:
            test_frames.append(df)

if test_frames:
    test_df = pd.concat(test_frames, ignore_index=True)
    print(f"\nTest result: {len(test_df):,} fire pixels from {len(test_frames)} granules")
    print(f"\nColumn summary:")
    cols = ['BT_I4', 'BT_I5', 'BTD', 'BTD_zscore', 'frp_mw', 'view_zen', 'confidence']
    print(test_df[cols].describe().round(2))

    # ── Verification checks ───────────────────────────────────────────────────
    print("\n── Verification checks ──")

    bt_ok  = (test_df['BT_I4'].dropna().between(208, 367)).mean()
    # FIX #3: threshold updated from >0 to >15 K.
    # 15 K is the documented lower bound for nominal-confidence pixels (BTD anomaly).
    # The fire mask BTD threshold for nominal confidence in the VNP14IMG algorithm
    # is defined at >15 K in the mid-infrared (I4) temperature anomaly.
    btd_ok = (test_df['BTD'].dropna() > 15).mean()
    frp_ok = (test_df['frp_mw'].dropna().between(0.08, 2100)).mean()

    # Saturation flag: I4 saturates at ~367 K on intense fires.
    # >5% saturation on a single day is worth investigating.
    sat_pct = (test_df['BT_I4'].dropna() >= 365).mean()

    # FIX #4: Check for unknown dates from metadata parse failures
    unknown_dt_pct = (test_df['datetime_utc'] == 'unknown').mean()

    print(f"BT_I4 in 208–367 K    : {bt_ok*100:.1f}%  (expect ~100%)")
    print(f"BTD > 15 K            : {btd_ok*100:.1f}%  (expect >50% on active fire day; <50% may indicate false alarms or smoldering)")
    print(f"FRP in 0.08–2100 MW   : {frp_ok*100:.1f}%  (expect ~100%)")
    print(f"BT_I4 saturated ≥365K : {sat_pct*100:.1f}%  (flag if >5%)")
    print(f"Unknown datetime_utc  : {unknown_dt_pct*100:.1f}%  (expect 0%; >0% means ERA5 co-location will fail for those rows)")

    # BTD_zscore sanity
    zscore_ok = test_df['BTD_zscore'].dropna()
    print(f"\nBTD_zscore stats:")
    print(f"  mean={zscore_ok.mean():.1f}  std={zscore_ok.std():.1f}  p95={zscore_ok.quantile(0.95):.1f}")
    print(f"  (expect mean > 5 and p95 >> mean for fire pixels)")

    print(f"\nDatetime UTC sample (first 5 rows):")
    print(test_df[['datetime_utc', 'date', 'granule_id']].head())
else:
    print("No fire pixels found - check FP_VARS paths match structure audit")

In [ ]:
# ── Visualise test day fire pixels ────────────────────────────────────────────
if test_frames:
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))

    sc0 = axes[0].scatter(test_df['longitude'], test_df['latitude'],
                          c=test_df['BT_I4'], cmap='inferno', s=4, alpha=0.7,
                          vmin=310, vmax=367)
    plt.colorbar(sc0, ax=axes[0], label='K')
    axes[0].set_title(f'BT_I4 (MWIR) - {TEST_DATE}')
    axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

    sc1 = axes[1].scatter(test_df['longitude'], test_df['latitude'],
                          c=test_df['BTD'], cmap='hot', s=4, alpha=0.7,
                          vmin=0, vmax=80)
    plt.colorbar(sc1, ax=axes[1], label='K')
    axes[1].set_title('BTD = BT_I4 − BT_I5')
    axes[1].set_xlabel('Longitude')

    # BTD_zscore - new in v2
    sc2 = axes[2].scatter(test_df['longitude'], test_df['latitude'],
                          c=test_df['BTD_zscore'].clip(-5, 50),
                          cmap='plasma', s=4, alpha=0.7)
    plt.colorbar(sc2, ax=axes[2], label='σ')
    axes[2].set_title('BTD_zscore (MAD-normalised)')
    axes[2].set_xlabel('Longitude')

    sc3 = axes[3].scatter(test_df['longitude'], test_df['latitude'],
                          c=np.log10(test_df['frp_mw'].clip(0.1)),
                          cmap='YlOrRd', s=4, alpha=0.7)
    plt.colorbar(sc3, ax=axes[3], label='log₁₀(MW)')
    axes[3].set_title('FRP (log scale)')
    axes[3].set_xlabel('Longitude')

    fig.suptitle(f'FireSight-IR | Module 1a v2 - VNP14IMG fire pixels, {TEST_DATE}', fontsize=10)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '01a_test_day_fire_pixels_v2.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Figure saved to Drive")

---
## Section 9 - Full batch processor

Processes all VIIRS granules for active fire days across 2018–2023.

**FIX #1 (critical):** Deduplication key changed from `['latitude', 'longitude', 'date']`  
to `['latitude', 'longitude', 'granule_id']`. The old key silently dropped valid  
multi-overpass observations (e.g., both the 10:30 and 22:30 overpasses of the same  
fire pixel on the same day). The corrected key removes only true swath-overlap  
duplicates - pixels appearing twice in consecutive scans within the same granule.  

**Runtime:** ~90–120 minutes total. 2020 and 2021 are the largest years.  
**If Colab disconnects:** Reconnect, rerun Sections 0–3, then rerun this cell. Completed years are skipped.

In [ ]:
def process_year(
    year: int,
    firms_dir: Path,
    output_dir: Path,
    bbox_dict: dict,
    bbox_tuple: tuple,
    short_name: str,
    version: str,
    fp_vars: dict,
    fp_fill: dict,
    min_firms_count: int = 10,
    granule_batch_size: int = 5,  # FIX #6: reduced from 10 to avoid memory pressure
) -> dict:
    """
    Process all active fire days for a single year.

    For each active fire day:
      1. Query CMR for VNP14IMG granules
      2. Stream granules from S3 in batches
      3. Extract fire pixel records
      4. Concatenate and write annual parquet to output_dir

    Deduplication
    -------------
    FIX #1: Deduplication is on ['latitude', 'longitude', 'granule_id'].
    This removes swath-overlap duplicates within a granule while preserving
    legitimate multi-overpass detections (different granule_id, same lat/lon/day).

    Returns
    -------
    dict: year, n_pixels, n_days, n_errors, n_unknown_dt
    """
    out_path = output_dir / f'viirs_fp_{year}.parquet'

    if out_path.exists():
        df_existing = pd.read_parquet(out_path)
        print(f"  {year}: {len(df_existing):>9,} pixels  [already on Drive - skipped]")
        return {'year': year, 'n_pixels': len(df_existing),
                'n_days': -1, 'n_errors': 0, 'n_unknown_dt': 0}

    active_days = get_active_fire_days(year, firms_dir, min_firms_count)
    if not active_days:
        print(f"  {year}: no active fire days found")
        return {'year': year, 'n_pixels': 0,
                'n_days': 0, 'n_errors': 0, 'n_unknown_dt': 0}

    year_frames = []
    n_errors = 0

    for day_str in tqdm(active_days, desc=f'{year}', unit='day', leave=False):
        try:
            day_granules = earthaccess.search_data(
                short_name   = short_name,
                version      = version,
                temporal     = (day_str, day_str),
                bounding_box = bbox_tuple,
                count        = 50,
            )

            if not day_granules:
                continue

            for i in range(0, len(day_granules), granule_batch_size):
                batch = day_granules[i:i + granule_batch_size]
                with earthaccess.open(batch) as fhs:
                    for granule, fh in zip(batch, fhs):
                        try:
                            df = extract_granule_fp(
                                fh, granule, bbox_dict, fp_vars, fp_fill
                            )
                            if len(df) > 0:
                                year_frames.append(df)
                        except Exception:
                            n_errors += 1

        except Exception as day_err:
            n_errors += 1
            if n_errors <= 3:
                print(f"  [WARN] {day_str}: {day_err}")

    if not year_frames:
        print(f"  {year}: no fire pixels extracted")
        return {'year': year, 'n_pixels': 0,
                'n_days': len(active_days), 'n_errors': n_errors, 'n_unknown_dt': 0}

    year_df = pd.concat(year_frames, ignore_index=True)

    # FIX #1: Deduplicate on granule_id, not date.
    # Preserves multi-overpass detections; removes only intra-granule swath duplicates.
    year_df = year_df.drop_duplicates(subset=['latitude', 'longitude', 'granule_id'])

    # FIX #4: Report unknown datetime rows - they will fail ERA5 co-location in 1b
    n_unknown_dt = (year_df['datetime_utc'] == 'unknown').sum()
    if n_unknown_dt > 0:
        print(f"  [WARN] {year}: {n_unknown_dt} pixels have datetime_utc='unknown' "
              f"- these will be excluded from ERA5 co-location in Module 1b")

    year_df.to_parquet(out_path, index=False)
    return {
        'year': year,
        'n_pixels':     len(year_df),
        'n_days':       len(active_days),
        'n_errors':     n_errors,
        'n_unknown_dt': int(n_unknown_dt),
    }


print("✓ process_year() defined")
print("  Deduplication key: ['latitude', 'longitude', 'granule_id']  (FIX #1)")
print("  Batch size: 5 granules  (reduced from 10 to manage Colab RAM)")

In [ ]:
# ── Run full extraction - 2018 to 2023 ────────────────────────────────────────
# Expect ~90–120 minutes total. 0% progress bars during S3 connection retries
# are normal - do not interrupt the cell.

extraction_log = []

for year in ALL_YEARS:
    flag = ' ← VAL YEAR' if year == VAL_YEAR else ''
    print(f"\n── {year}{flag} ──")
    summary = process_year(
        year             = year,
        firms_dir        = FIRMS_DIR,
        output_dir       = VIIRS_DIR,
        bbox_dict        = BBOX_DICT,
        bbox_tuple       = BBOX_TUPLE,
        short_name       = VIIRS_SHORT_NAME,
        version          = VIIRS_VERSION,
        fp_vars          = FP_VARS,
        fp_fill          = FP_FILL,
        min_firms_count  = MIN_FIRMS_COUNT,
    )
    extraction_log.append(summary)
    if summary['n_days'] != -1:
        print(
            f"  {year}: {summary['n_pixels']:>9,} fire pixels | "
            f"{summary['n_days']} active days | "
            f"{summary['n_errors']} errors | "
            f"{summary['n_unknown_dt']} unknown-datetime rows"
        )

print("\n✓ Full extraction complete")

---
## Section 10 - Output verification

In [ ]:
# ── Verification checklist ────────────────────────────────────────────────────
print("═══ FireSight-IR Module 1a v2 - Output Verification ═══\n")

# Check 1: Year-by-year pixel counts
EXPECTED_PIXELS = {
    2018: (150_000, 250_000),
    2019: (40_000,  90_000),
    2020: (250_000, 450_000),
    2021: (250_000, 450_000),
    2022: (60_000,  150_000),
    2023: (40_000,  130_000),
}

print(f"{'Year':<8} {'Pixels':>12} {'Status':<10} {'Expected range':<25} {'Note'}")
print('─' * 80)

pixel_counts = {}
for yr in ALL_YEARS:
    path = VIIRS_DIR / f'viirs_fp_{yr}.parquet'
    if not path.exists():
        print(f"{yr:<8} {'MISSING':>12} {'FAIL':<10}")
        continue

    df = pd.read_parquet(path)
    n = len(df)
    pixel_counts[yr] = n
    lo, hi = EXPECTED_PIXELS.get(yr, (0, 9e9))
    ok   = 'PASS' if lo <= n <= hi else 'CHECK'
    note = '← VAL YEAR' if yr == VAL_YEAR else ''
    print(f"{yr:<8} {n:>12,} {ok:<10} {lo:,} – {hi:,}        {note}")

    if yr == 2020:
        bt_ok   = df['BT_I4'].dropna().between(208, 367).mean()
        # FIX #3: updated threshold from >0 to >15
        btd_ok  = (df['BTD'].dropna() > 15).mean()
        frp_ok  = df['frp_mw'].dropna().between(0.08, 2100).mean()
        sat_pct = (df['BT_I4'].dropna() >= 365).mean()
        unk_dt  = (df['datetime_utc'] == 'unknown').mean()
        print(f"  2020 range checks:")
        print(f"    BT_I4 in 208–367 K        : {bt_ok*100:.1f}%  (expect ~100%)")
        print(f"    BTD > 15 K (nominal conf) : {btd_ok*100:.1f}%  (expect >50%)")
        print(f"    FRP in range              : {frp_ok*100:.1f}%  (expect ~100%)")
        print(f"    BT_I4 saturated ≥365 K    : {sat_pct*100:.1f}%  (flag if >5%)")
        print(f"    Unknown datetime_utc      : {unk_dt*100:.2f}% (expect ~0%)")

# Check 2: Total training pixels
total_pixels = sum(
    pixel_counts.get(yr, 0) for yr in TRAIN_YEARS
)
print(f"\nTotal training pixels (2018–2022): {total_pixels:,}")
print(f"  Expect >1,000,000 - {'PASS' if total_pixels > 1_000_000 else 'CHECK'}")

# Check 3: 2020 and 2021 are the largest years
if pixel_counts:
    top2 = sorted(pixel_counts, key=pixel_counts.get, reverse=True)[:2]
    extreme_ok = set(top2) <= {2020, 2021}
    print(f"\nTop 2 years by pixel count: {top2}")
    print(f"  2020 and 2021 highest - {'PASS' if extreme_ok else 'CHECK (unexpected)'}")

# Check 4: BTD_zscore column present and non-trivial
if pixel_counts:
    sample_path = VIIRS_DIR / f'viirs_fp_2020.parquet'
    if sample_path.exists():
        df = pd.read_parquet(sample_path, columns=['BTD_zscore', 'datetime_utc'])
        zscore_nonnan = df['BTD_zscore'].dropna().shape[0] / len(df)
        print(f"\nBTD_zscore completeness (2020): {zscore_nonnan*100:.1f}%  (expect >90%)")
        print(f"  {'PASS' if zscore_nonnan > 0.9 else 'CHECK'}")

# Check 5: Drive file count
n_files = len(list(FIRMS_DIR.glob('*.parquet'))) + len(list(VIIRS_DIR.glob('*.parquet')))
print(f"\nParquet files on Drive: {n_files}  (expect 12 = 6 FIRMS + 6 VIIRS)")
print(f"  {'PASS' if n_files == 12 else f'CHECK - {n_files}/12 files present'}")

In [ ]:
# ── Annual pixel count chart ───────────────────────────────────────────────────
if pixel_counts:
    fig, ax = plt.subplots(figsize=(9, 4))
    years_x = list(pixel_counts.keys())
    counts  = list(pixel_counts.values())
    colors  = ['#A32D2D' if yr == VAL_YEAR else '#378ADD' for yr in years_x]

    bars = ax.bar(years_x, counts, color=colors, alpha=0.85,
                  edgecolor='white', linewidth=0.5)

    for bar, n in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
                f'{n/1e3:.0f}k', ha='center', va='bottom', fontsize=9)

    ax.set_xlabel('Year')
    ax.set_ylabel('Fire pixels')
    ax.set_title(
        'FireSight-IR | Module 1a v2 - VNP14IMG fire pixels per year\n'
        'Blue = training | Red = validation (held out)'
    )
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k')
    )

    annotations = {
        2018: 'Camp Fire\nMendocino Complex',
        2020: 'Creek Fire\nAugust Complex\n(record season)',
        2021: 'Dixie Fire\nCaldor Fire',
    }
    for yr, label in annotations.items():
        if yr in pixel_counts:
            ax.annotate(
                label, xy=(yr, pixel_counts[yr]),
                xytext=(yr, pixel_counts[yr] + max(counts) * 0.06),
                ha='center', fontsize=7.5, color='#333333',
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.8)
            )

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '01a_pixel_counts_v2.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Figure saved to Drive")

---
## Section 11 - FIRMS type column audit (label construction preview)

In [ ]:
# ── Audit FIRMS type distribution across all years ────────────────────────────
print("FIRMS type column - label construction preview")
print("type=0: vegetation fire → positive wildfire label")
print("type=2: static land source (gas flares, industrial) → false-alarm training")
print("type=3: offshore → exclude from training")
print()

type_rows = []
for yr in ALL_YEARS:
    path = FIRMS_DIR / f'firms_viirs_snpp_{yr}.parquet'
    if not path.exists():
        continue
    df = pd.read_parquet(path, columns=['type'])
    counts = df['type'].value_counts().to_dict()
    total  = len(df)
    type_rows.append({
        'year':    yr,
        'type_0':  counts.get(0, 0),
        'type_2':  counts.get(2, 0),
        'type_3':  counts.get(3, 0),
        'total':   total,
        'pct_veg': 100 * counts.get(0, 0) / max(total, 1),
    })

type_df = pd.DataFrame(type_rows)
print(type_df.to_string(index=False))

total_type2 = type_df['type_2'].sum()
print(f"\nTotal type=2 detections (false-alarm pool): {total_type2:,}")
print(
    f"  {'Good - sufficient false-alarm training examples' if total_type2 > 1000 else 'Low - may need to supplement with DNB/OSM masks in Module 2'}"
)

---
## Section 12 - Summary

### Output files on Google Drive

| File | Path | Description |
|---|---|---|
| `firms_viirs_snpp_{YEAR}.parquet` | `data/raw/firms/` | Ground truth labels - one row per fire detection |
| `viirs_fp_{YEAR}.parquet` | `data/processed/viirs_fp/` | Primary ML inputs - one row per fire pixel |
| `01a_test_day_fire_pixels_v2.png` | `figures/` | BT_I4 / BTD / BTD_zscore / FRP for 2020-09-05 |
| `01a_pixel_counts_v2.png` | `figures/` | Annual fire pixel counts |

### Column reference for viirs_fp parquet files

| Column | Source | Description |
|---|---|---|
| `BT_I4`, `BT_I5` | VNP14IMG | Brightness temperature in K - FireSat-analog MWIR/LWIR |
| `BT_I4_bg`, `BT_I5_bg` | VNP14IMG | Local background mean BT |
| `MAD_I4`, `MAD_I5` | VNP14IMG | Local background mean absolute deviation |
| `frp_mw` | VNP14IMG | Fire radiative power in MW |
| `confidence` | VNP14IMG | 7=low / 8=nominal / 9=high (verify in Section 5 audit) |
| `sol_zen`, `view_zen` | VNP14IMG | Solar and view zenith angles - critical for Beer-Lambert |
| `BTD` | derived | BT_I4 − BT_I5 - primary fire spectral signal |
| `BT_I4_anom` | derived | BT_I4 − BT_I4_bg - fire intensity above ambient |
| `BTD_anom` | derived | BTD − BT_diff_bg - fire spectral contrast above background |
| `BTD_zscore` | derived | (BTD − BT_diff_bg) / (MAD_I4 + ε) - MAD-normalised contrast |
| `datetime_utc` | granule metadata | UTC timestamp 'YYYY-MM-DDTHH:MM:SS' - **required for Module 1b ERA5 co-location** |
| `date` | granule metadata | 'YYYY-MM-DD' - for day-level grouping |
| `granule_id` | granule metadata | Unique granule identifier - used for deduplication |

### v1 → v2 changes summary

| Fix | Where | Impact |
|---|---|---|
| Dedup key: `granule_id` not `date` | `process_year()` | Preserves multi-overpass fire pixels; critical for nighttime fire coverage |
| Fill masking from audit, not hardcoded | `extract_granule_fp()` | Prevents corrupt derived features from unmask background variables |
| BTD threshold: `>15 K` not `>0` | Verification cells | Aligned with documented nominal-confidence boundary |
| Explicit unknown-datetime warning | `_parse_granule_datetime()` | Surfaces ERA5 co-location failures before Module 1b runs |
| `datetime_utc` column | `extract_granule_fp()` | ERA5 co-location in Module 1b requires UTC hour resolution |
| `BTD_zscore` feature | `extract_granule_fp()` | MAD-normalised contrast; more stable ML input than raw BTD_anom |

---

### What comes next - Module 1b

Module 1b downloads ERA5 atmospheric profiles co-located with the fire pixels produced here.  
For each fire detection location and **UTC timestamp**, it retrieves temperature profile,  
specific humidity, and PBL height for the Beer-Lambert transmittance calculations.

**Before Module 1b:** Register for a free Copernicus CDS account at  
https://cds.climate.copernicus.eu/ and install your CDS API key.